In [ ]:
import requests
import pandas as pd
import json
import time
from getpass import getpass
from pathlib import Path

In [ ]:
API_KEY = getpass("Enter API key: ")
BASE_URL = "https://www.steamwebapi.com/steam/api/apps"
headers = {
    "X-Api-Key": API_KEY
}

In [ ]:
def get_all_games(api_key, limit=500, max_calls=40, delay=7.25):
    """
    Download up to max_calls pages of the Steam game catalog
    --
    Parameters:
    api_key = string (SteamWebAPI key)
    limit = int (number of games per request)
    max_calls = int (maximum number of requests)
    delay = float (seconds to wait between requests)
    --
    Returns:
    list containing game dictionaries
    """

    all_games = []
    cursor = None
    call_number = 0

    while call_number < max_calls:
        params = {
            "limit": limit,
            "sort": "appid",
            "order": "asc"
        }

        # Add cursor after first request
        if cursor is not None:
            params["cursor"] = cursor

        call_number += 1

        try:
            response = requests.get(
                BASE_URL,
                headers=headers,
                params=params,
                timeout=60
            )

            print(
                f"API call {call_number}/{max_calls} "
                f"- HTTP {response.status_code}"
            )
    
            # Stop if the request failed
            if response.status_code != 200:
                print("Request failed.")
                print("Returning games collected so far.")
                break
    
            # Get games from this page
            data = response.json()
            games = data.get("data", [])
            all_games.extend(games)
            print(
                f"    Games this gall: {len(games):,}"
            )
            print(
                f"    Total collected: {len(all_games):,}"
            )
    
            # Get next cursor
            pagination = data.get("pagination", {})
            next_cursor = pagination.get("nextcursor")
    
            # Stop when no next page exists
            if not next_cursor:
                print("Reached final page.")
                break
    
            cursor = next_cursor
    
            # Small pause to not overwhelm API
            time.sleep(delay)
    
        except requests.RequestException as e:
            print(f"Request error: {e}")
            print(f"Returning games collected so far.")
            break
    
        except Exception as e:
            print(f"Unexpected error: {e}")
            print(f"Returning games collected so far.")
            break

    if call_number >= max_calls:
        print(
            f"\nReached the maximum of {max_calls} API calls."
        )
    print(
        f"\nFinished. Successfully collected "
        f"{len(all_games):,} games using"
        f"{call_number} API calls."
    )

    return all_games

games = get_all_games(API_KEY)

In [ ]:
# ORIGINAL DOWNLOADED DATA
Path("data/raw").mkdir(parents=True, exist_ok=True)

raw_file = Path("data/raw/steam_games_raw.json")

with open(raw_file, "w", encoding="utf-8") as f:
    json.dump(games, f, ensure_ascii=False, indent=2)

print(f"Saved raw data to: {raw_file}")

In [ ]:
# CONVERT TO PANDAS
df = pd.json_normalize(games)
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()
df.info()
df.columns.tolist()

In [ ]:
# SAVE AS CSV
Path("data/processed").mkdir(parents=True, exist_ok=True)

df.to_csv(
    "data/processed/steam_games.csv",
    index=False
)

print("CSV saved.")